# Experiment 12: Kubernetes Setup for ML Application

**Objective:**
- Create Kubernetes manifests for ML application deployment
- Set up Deployment, Service, ConfigMap, and HPA
- Deploy with monitoring stack on Kubernetes
- Implement rolling updates and health checks

**Prerequisites:** Run Experiments 1-5 (Model, API, Docker)

## Step 1: Install Required Tools

In [ ]:
# Install Kubernetes Python client
!pip install kubernetes pyyaml

In [ ]:
import subprocess, os, json, yaml

# Check if kubectl and docker are available
for cmd in ['kubectl', 'docker']:
    try:
        result = subprocess.run([cmd, 'version', '--short'], capture_output=True, text=True, timeout=5)
        print(f"{cmd}: {result.stdout.strip().split(chr(10))[0]}")
    except:
        print(f"{cmd}: Not found. Install from:")
        if cmd == 'kubectl':
            print("  brew install kubectl  (macOS)")
            print("  https://kubernetes.io/docs/tasks/tools/")
        else:
            print("  https://docs.docker.com/get-docker/")

# Check for minikube or kind
for tool in ['minikube', 'kind']:
    try:
        result = subprocess.run([tool, 'version'], capture_output=True, text=True, timeout=5)
        print(f"{tool}: {result.stdout.strip().split(chr(10))[0]}")
    except:
        print(f"{tool}: Not found")

print("\nTo install Minikube: brew install minikube")
print("To install Kind: brew install kind")

## Step 2: Create Kubernetes Manifests Directory

In [ ]:
os.makedirs('k8s', exist_ok=True)
print("Created: k8s/ directory for Kubernetes manifests")

## Step 3: Create Namespace

In [ ]:
namespace = {
    "apiVersion": "v1",
    "kind": "Namespace",
    "metadata": {
        "name": "ml-churn",
        "labels": {
            "app": "churn-predictor",
            "project": "mlops"
        }
    }
}

with open('k8s/namespace.yaml', 'w') as f:
    yaml.dump(namespace, f, default_flow_style=False)

print("Created: k8s/namespace.yaml")

## Step 4: Create ConfigMap and Secrets

In [ ]:
configmap = {
    "apiVersion": "v1",
    "kind": "ConfigMap",
    "metadata": {
        "name": "ml-api-config",
        "namespace": "ml-churn"
    },
    "data": {
        "APP_NAME": "churn-predictor",
        "APP_VERSION": "1.0.0",
        "LOG_LEVEL": "INFO",
        "MODEL_PATH": "/app/model_artifacts",
        "WORKERS": "2"
    }
}

with open('k8s/configmap.yaml', 'w') as f:
    yaml.dump(configmap, f, default_flow_style=False)

# Secrets (base64 encoded in real scenarios)
import base64

secret = {
    "apiVersion": "v1",
    "kind": "Secret",
    "metadata": {
        "name": "ml-api-secrets",
        "namespace": "ml-churn"
    },
    "type": "Opaque",
    "data": {
        "JWT_SECRET_KEY": base64.b64encode(b"mlops-super-secret-key-2024").decode(),
        "API_KEY_1": base64.b64encode(b"mlops-api-key-001").decode(),
        "API_KEY_2": base64.b64encode(b"mlops-api-key-002").decode()
    }
}

with open('k8s/secrets.yaml', 'w') as f:
    yaml.dump(secret, f, default_flow_style=False)

print("Created: k8s/configmap.yaml")
print("Created: k8s/secrets.yaml")

## Step 5: Create Deployment

In [ ]:
deployment_yaml = '''apiVersion: apps/v1
kind: Deployment
metadata:
  name: ml-api
  namespace: ml-churn
  labels:
    app: ml-api
    version: v1
spec:
  replicas: 3
  selector:
    matchLabels:
      app: ml-api
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 0
  template:
    metadata:
      labels:
        app: ml-api
        version: v1
      annotations:
        prometheus.io/scrape: "true"
        prometheus.io/port: "8000"
        prometheus.io/path: "/metrics"
    spec:
      containers:
        - name: ml-api
          image: ml-churn-api:latest
          imagePullPolicy: IfNotPresent
          ports:
            - containerPort: 8000
              name: http
          envFrom:
            - configMapRef:
                name: ml-api-config
          env:
            - name: JWT_SECRET_KEY
              valueFrom:
                secretKeyRef:
                  name: ml-api-secrets
                  key: JWT_SECRET_KEY
          resources:
            requests:
              cpu: "100m"
              memory: "256Mi"
            limits:
              cpu: "500m"
              memory: "512Mi"
          livenessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 15
            periodSeconds: 20
            timeoutSeconds: 5
            failureThreshold: 3
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 10
            periodSeconds: 10
            timeoutSeconds: 3
          startupProbe:
            httpGet:
              path: /health
              port: 8000
            failureThreshold: 30
            periodSeconds: 5
      terminationGracePeriodSeconds: 30
'''

with open('k8s/deployment.yaml', 'w') as f:
    f.write(deployment_yaml)

print("Created: k8s/deployment.yaml")
print("  - 3 replicas with RollingUpdate strategy")
print("  - Liveness, readiness, and startup probes")
print("  - Resource requests/limits configured")
print("  - Prometheus scrape annotations")

## Step 6: Create Service

In [ ]:
service_yaml = '''apiVersion: v1
kind: Service
metadata:
  name: ml-api-service
  namespace: ml-churn
  labels:
    app: ml-api
spec:
  type: LoadBalancer
  selector:
    app: ml-api
  ports:
    - name: http
      port: 80
      targetPort: 8000
      protocol: TCP
---
apiVersion: v1
kind: Service
metadata:
  name: ml-api-nodeport
  namespace: ml-churn
  labels:
    app: ml-api
spec:
  type: NodePort
  selector:
    app: ml-api
  ports:
    - name: http
      port: 8000
      targetPort: 8000
      nodePort: 30080
      protocol: TCP
'''

with open('k8s/service.yaml', 'w') as f:
    f.write(service_yaml)

print("Created: k8s/service.yaml")
print("  - LoadBalancer service on port 80")
print("  - NodePort service on port 30080")

## Step 7: Create Horizontal Pod Autoscaler (HPA)

In [ ]:
hpa_yaml = '''apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: ml-api-hpa
  namespace: ml-churn
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: ml-api
  minReplicas: 2
  maxReplicas: 10
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 70
    - type: Resource
      resource:
        name: memory
        target:
          type: Utilization
          averageUtilization: 80
  behavior:
    scaleUp:
      stabilizationWindowSeconds: 60
      policies:
        - type: Pods
          value: 2
          periodSeconds: 60
    scaleDown:
      stabilizationWindowSeconds: 300
      policies:
        - type: Pods
          value: 1
          periodSeconds: 120
'''

with open('k8s/hpa.yaml', 'w') as f:
    f.write(hpa_yaml)

print("Created: k8s/hpa.yaml")
print("  - Min: 2 replicas, Max: 10 replicas")
print("  - Scale up at 70% CPU / 80% memory")
print("  - Scale-down stabilization: 5 minutes")

## Step 8: Create Ingress

In [ ]:
ingress_yaml = '''apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: ml-api-ingress
  namespace: ml-churn
  annotations:
    nginx.ingress.kubernetes.io/rewrite-target: /
    nginx.ingress.kubernetes.io/rate-limit: "100"
    nginx.ingress.kubernetes.io/rate-limit-window: "1m"
spec:
  ingressClassName: nginx
  rules:
    - host: ml-api.local
      http:
        paths:
          - path: /
            pathType: Prefix
            backend:
              service:
                name: ml-api-service
                port:
                  number: 80
'''

with open('k8s/ingress.yaml', 'w') as f:
    f.write(ingress_yaml)

print("Created: k8s/ingress.yaml")
print("  - Host: ml-api.local")
print("  - Rate limiting: 100 req/min")

## Step 9: Create Monitoring Stack on K8s (Prometheus + Grafana)

In [ ]:
monitoring_yaml = '''# Prometheus ConfigMap
apiVersion: v1
kind: ConfigMap
metadata:
  name: prometheus-config
  namespace: ml-churn
data:
  prometheus.yml: |
    global:
      scrape_interval: 15s
    scrape_configs:
      - job_name: ml-api
        kubernetes_sd_configs:
          - role: pod
            namespaces:
              names: [ml-churn]
        relabel_configs:
          - source_labels: [__meta_kubernetes_pod_annotation_prometheus_io_scrape]
            action: keep
            regex: true
          - source_labels: [__meta_kubernetes_pod_annotation_prometheus_io_port]
            action: replace
            target_label: __address__
            regex: (.+)
            replacement: ${1}:8000
---
# Prometheus Deployment
apiVersion: apps/v1
kind: Deployment
metadata:
  name: prometheus
  namespace: ml-churn
spec:
  replicas: 1
  selector:
    matchLabels:
      app: prometheus
  template:
    metadata:
      labels:
        app: prometheus
    spec:
      containers:
        - name: prometheus
          image: prom/prometheus:latest
          ports:
            - containerPort: 9090
          volumeMounts:
            - name: config
              mountPath: /etc/prometheus
          resources:
            requests:
              cpu: "100m"
              memory: "256Mi"
            limits:
              cpu: "500m"
              memory: "512Mi"
      volumes:
        - name: config
          configMap:
            name: prometheus-config
---
apiVersion: v1
kind: Service
metadata:
  name: prometheus
  namespace: ml-churn
spec:
  type: NodePort
  selector:
    app: prometheus
  ports:
    - port: 9090
      targetPort: 9090
      nodePort: 30090
---
# Grafana Deployment
apiVersion: apps/v1
kind: Deployment
metadata:
  name: grafana
  namespace: ml-churn
spec:
  replicas: 1
  selector:
    matchLabels:
      app: grafana
  template:
    metadata:
      labels:
        app: grafana
    spec:
      containers:
        - name: grafana
          image: grafana/grafana:latest
          ports:
            - containerPort: 3000
          env:
            - name: GF_SECURITY_ADMIN_USER
              value: "admin"
            - name: GF_SECURITY_ADMIN_PASSWORD
              value: "admin123"
          resources:
            requests:
              cpu: "100m"
              memory: "128Mi"
            limits:
              cpu: "250m"
              memory: "256Mi"
---
apiVersion: v1
kind: Service
metadata:
  name: grafana
  namespace: ml-churn
spec:
  type: NodePort
  selector:
    app: grafana
  ports:
    - port: 3000
      targetPort: 3000
      nodePort: 30030
'''

with open('k8s/monitoring.yaml', 'w') as f:
    f.write(monitoring_yaml)

print("Created: k8s/monitoring.yaml")
print("  - Prometheus (NodePort 30090)")
print("  - Grafana (NodePort 30030)")

## Step 10: Create Deployment Script
code

deploy_script = '''#!/bin/bash
set -e

echo "========================================"
echo "  ML Churn Predictor - K8s Deployment"
echo "========================================"

# Check prerequisites
command -v kubectl >/dev/null 2>&1 || { echo "kubectl not found. Install it first."; exit 1; }
command -v docker >/dev/null 2>&1 || { echo "docker not found. Install it first."; exit 1; }

# Variables
IMAGE_NAME="ml-churn-api"
IMAGE_TAG="latest"

# Step 1: Build Docker image
echo "\n[1/5] Building Docker image..."
docker build -t ${IMAGE_NAME}:${IMAGE_TAG} .

# Step 2: Load image into local cluster (for minikube/kind)
echo "\n[2/5] Loading image into cluster..."
if command -v minikube &> /dev/null; then
    eval $(minikube docker-env)
    docker build -t ${IMAGE_NAME}:${IMAGE_TAG} .
    echo "  Image loaded into Minikube"
elif command -v kind &> /dev/null; then
    kind load docker-image ${IMAGE_NAME}:${IMAGE_TAG}
    echo "  Image loaded into Kind"
else
    echo "  Warning: No local cluster tool found (minikube/kind)"
fi

# Step 3: Apply Kubernetes manifests
echo "\n[3/5] Applying Kubernetes manifests..."
kubectl apply -f k8s/namespace.yaml
kubectl apply -f k8s/configmap.yaml
kubectl apply -f k8s/secrets.yaml
kubectl apply -f k8s/deployment.yaml
kubectl apply -f k8s/service.yaml
kubectl apply -f k8s/hpa.yaml
kubectl apply -f k8s/ingress.yaml
kubectl apply -f k8s/monitoring.yaml

# Step 4: Wait for rollout
echo "\n[4/5] Waiting for deployment rollout..."
kubectl rollout status deployment/ml-api -n ml-churn --timeout=120s

# Step 5: Display status
echo "\n[5/5] Deployment Status"
echo "---"
kubectl get all -n ml-churn
echo "---"
echo "\n✅ Deployment complete!"
echo "\nAccess points:"

if command -v minikube &> /dev/null; then
    MINIKUBE_IP=$(minikube ip)
    echo "  ML API:      http://${MINIKUBE_IP}:30080"
    echo "  Prometheus:  http://${MINIKUBE_IP}:30090"
    echo "  Grafana:     http://${MINIKUBE_IP}:30030"
else
    echo "  ML API:      kubectl port-forward svc/ml-api-service -n ml-churn 8000:80"
    echo "  Prometheus:  kubectl port-forward svc/prometheus -n ml-churn 9090:9090"
    echo "  Grafana:     kubectl port-forward svc/grafana -n ml-churn 3000:3000"
fi
'''

with open('k8s/deploy.sh', 'w') as f:
    f.write(deploy_script)

os.chmod('k8s/deploy.sh', 0o755)
print("Created: k8s/deploy.sh (executable)")
markdown
## Step 11: Create Cleanup Script
code

cleanup_script = '''#!/bin/bash
echo "Cleaning up ML Churn K8s resources..."

kubectl delete -f k8s/monitoring.yaml --ignore-not-found
kubectl delete -f k8s/ingress.yaml --ignore-not-found
kubectl delete -f k8s/hpa.yaml --ignore-not-found
kubectl delete -f k8s/service.yaml --ignore-not-found
kubectl delete -f k8s/deployment.yaml --ignore-not-found
kubectl delete -f k8s/secrets.yaml --ignore-not-found
kubectl delete -f k8s/configmap.yaml --ignore-not-found
kubectl delete -f k8s/namespace.yaml --ignore-not-found

echo "✅ Cleanup complete!"
'''

with open('k8s/cleanup.sh', 'w') as f:
    f.write(cleanup_script)

os.chmod('k8s/cleanup.sh', 0o755)
print("Created: k8s/cleanup.sh (executable)")
markdown
## Step 12: Validate Manifests with Python
code

# Validate all YAML files
import glob

print("Validating Kubernetes manifests...")
print("=" * 50)

k8s_files = sorted(glob.glob('k8s/*.yaml'))
for filepath in k8s_files:
    try:
        with open(filepath) as f:
            docs = list(yaml.safe_load_all(f))
        for doc in docs:
            if doc:
                kind = doc.get('kind', 'Unknown')
                name = doc.get('metadata', {}).get('name', 'unknown')
                ns = doc.get('metadata', {}).get('namespace', 'default')
                print(f"  ✅ {os.path.basename(filepath):25s} → {kind:30s} {name} ({ns})")
    except Exception as e:
        print(f"  ❌ {os.path.basename(filepath):25s} → Error: {e}")

print(f"\nTotal files: {len(k8s_files)}")
markdown
## Step 13: Deployment Commands Reference
code

print("=" * 60)
print("KUBERNETES DEPLOYMENT GUIDE")
print("=" * 60)
print("")
print("── Option A: Using Minikube ──────────────────────")
print("  minikube start --cpus=2 --memory=4096")
print("  minikube addons enable metrics-server")
print("  minikube addons enable ingress")
print("  eval $(minikube docker-env)")
print("  bash k8s/deploy.sh")
print("  minikube dashboard")
print("")
print("── Option B: Using Kind ─────────────────────────")
print("  kind create cluster --name ml-cluster")
print("  docker build -t ml-churn-api:latest .")
print("  kind load docker-image ml-churn-api:latest --name ml-cluster")
print("  bash k8s/deploy.sh")
print("")
print("── Useful Commands ─────────────────────────────")
print("  kubectl get pods -n ml-churn                       # List pods")
print("  kubectl logs -f deployment/ml-api -n ml-churn      # Stream logs")
print("  kubectl describe pod <pod-name> -n ml-churn        # Pod details")
print("  kubectl get hpa -n ml-churn                        # HPA status")
print("  kubectl top pods -n ml-churn                       # Resource usage")
print("  kubectl scale deploy/ml-api --replicas=5 -n ml-churn  # Scale manually")
print("")
print("── Port Forwarding ─────────────────────────────")
print("  kubectl port-forward svc/ml-api-service -n ml-churn 8000:80")
print("  kubectl port-forward svc/prometheus -n ml-churn 9090:9090")
print("  kubectl port-forward svc/grafana -n ml-churn 3000:3000")
print("")
print("── Rolling Update ──────────────────────────────")
print("  kubectl set image deploy/ml-api ml-api=ml-churn-api:v2 -n ml-churn")
print("  kubectl rollout status deploy/ml-api -n ml-churn")
print("  kubectl rollout undo deploy/ml-api -n ml-churn     # Rollback")
print("")
print("── Cleanup ─────────────────────────────────────")
print("  bash k8s/cleanup.sh")
print("  minikube delete  # or: kind delete cluster")
markdown
## Step 14: Summary
code

print("=" * 60)
print("KUBERNETES SETUP COMPLETE")
print("=" * 60)
print("")
print("Kubernetes manifests created in k8s/:")
print("  - namespace.yaml    → ml-churn namespace")
print("  - configmap.yaml    → App configuration")
print("  - secrets.yaml      → JWT & API keys")
print("  - deployment.yaml   → 3 replicas, health probes, rolling update")
print("  - service.yaml      → LoadBalancer + NodePort services")
print("  - hpa.yaml          → Auto-scaling (2-10 pods)")
print("  - ingress.yaml      → Nginx ingress with rate limiting")
print("  - monitoring.yaml   → Prometheus + Grafana on K8s")
print("  - deploy.sh         → Automated deployment script")
print("  - cleanup.sh        → Cleanup script")
print("")
print("Architecture:")
print("  [Ingress] → [Service/LB] → [3x ML API Pods]")
print("                               ↑ HPA (auto-scale)")
print("  [Prometheus] ← scrape ← [Pods /metrics]")
print("  [Grafana] → [Prometheus]")
print("")
print("\n✅ Experiment 12 Complete!")
print("━" * 60)
print("🎉 ALL 12 MLOPS EXPERIMENTS COMPLETED!")
print("━" * 60)